# evaluation

Loads `data/processed/playlist_features.csv` and `data/processed/growth_targets.csv`, performs a time holdout on `snapshot_date`, and writes `data/processed/eval_predictions.csv` + `data/processed/eval_metrics.json`.

In [ ]:
import os, json
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score


### Load data

In [ ]:
pf_path = "data/processed/playlist_features.csv"
gt_path = "data/processed/growth_targets.csv"

pf = pd.read_csv(pf_path)
gt = pd.read_csv(gt_path)

gt["snapshot_date"] = pd.to_datetime(gt["snapshot_date"], errors="coerce")

df=gt.merge(pf, on="playlist_id", how="left")

print("playlist_features:", pf.shape)
print("growth_targets:", gt.shape)
print("merged:", df.shape)
print("unique playlists:",df["playlist_id"].nunique())
print("unique dates:", df["snapshot_date"].nunique())
print("rows:", len(df),"cols:",len(df.columns))


In [ ]:
target = "delta_log_followers" if "delta_log_followers" in df.columns else "delta_followers"
if target not in df.columns:
    raise ValueError("Error: No target column found --> Expected delta_log_followers or delta_followers.")

exclude = {
    "playlist_id","playlist_name","snapshot_date",
    "followers","followers_prev","followers_next",
    "delta_followers","delta_log_followers",
    "log_followers","log_followers_prev",
}
feature_cols = [c for c in df.columns if c not in exclude]
X = df[feature_cols].select_dtypes(include=["number"]).copy()
y = df[target].astype(float).copy()

print("target:", target)
print("X:",X.shape, "y:", y.shape)


In [ ]:
dates = np.array(sorted(df["snapshot_date"].dropna().unique()))
if len(dates) < 2:
    raise ValueError("Error: need at least 2 different snapshot_date values for time holdout")

n_test_dates = max(1, int(round(0.2 * len(dates))))
test_dates = set(dates[-n_test_dates:])

is_test = df["snapshot_date"].isin(test_dates)
train_idx = df.index[~is_test].to_numpy()
test_idx  = df.index[is_test].to_numpy()

print("# different dates:", len(dates))
print("Test dates:", [pd.Timestamp(d).date().isoformat() for d in sorted(test_dates)])
print("Train rows:", len(train_idx), "test rows:", len(test_idx))
if len(train_idx) == 0 or len(test_idx) == 0:
    raise ValueError("Error: time split produced empty train or test set.")


In [ ]:
def spearman_corr(a, b):
    a = pd.Series(a).rank(method="average")
    b = pd.Series(b).rank(method="average")
    if len(a) < 2:
        return np.nan
    return float(a.corr(b, method="pearson"))

def topk_capture(y_true, y_pred, frac=0.1):
    y_true = pd.Series(y_true)
    y_pred = pd.Series(y_pred)
    n = len(y_true)
    if n<5:
        return np.nan
    k = max(1, int(round(frac * n)))
    true_top = set(y_true.nlargest(k).index)
    pred_top = set(y_pred.nlargest(k).index)
    return float(len(true_top & pred_top) / len(true_top))


In [ ]:
X_train,y_train = X.loc[train_idx], y.loc[train_idx]
X_test, y_test  = X.loc[test_idx],  y.loc[test_idx]

pred_zero = np.zeros(len(y_test), dtype=float)
pred_mean = np.full(len(y_test), float(np.mean(y_train)), dtype=float)

baseline_rows = []
for name, pred in {"baseline_zero": pred_zero, "baseline_mean": pred_mean}.items():
    baseline_rows.append({
        "model": name,
        "mae": mean_absolute_error(y_test, pred),
        "rmse": root_mean_squared_error(y_test, pred),
        "r2": (np.nan if len(y_test) < 2 else r2_score(y_test, pred)),
        "spearman": spearman_corr(y_test, pred),
        "top10_capture": topk_capture(y_test, pred, 0.10),
    })
baseline_metrics = pd.DataFrame(baseline_rows).sort_values("mae")
display(baseline_metrics)


In [ ]:
models = {
    "ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]),
    "linear": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
}

rows = []
pred_dfs = []

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rows.append({
        "model": name,
        "mae": mean_absolute_error(y_test, pred),
        "rmse": root_mean_squared_error(y_test, pred),
        "r2": (np.nan if len(y_test) < 2 else r2_score(y_test, pred)),
        "spearman": spearman_corr(y_test, pred),
        "top10_capture": topk_capture(y_test, pred, 0.10),
    })
    pred_dfs.append(pd.DataFrame({
        "model": name,
        "playlist_id": df.loc[test_idx, "playlist_id"].values,
        "snapshot_date": df.loc[test_idx, "snapshot_date"].dt.date.astype(str).values,
        "y_true": y_test.values,
        "y_pred": pred,
    }))

metrics = pd.DataFrame(rows).sort_values("mae")
display(metrics)
pred_out = pd.concat(pred_dfs, ignore_index=True)
display(pred_out.sort_values("y_pred", ascending=False).head(20))


In [ ]:
os.makedirs("data/processed", exist_ok=True)

pred_path = "data/processed/eval_predictions.csv"
metrics_path = "data/processed/eval_metrics.json"

pred_out.to_csv(pred_path, index=False)

p = {
    "target": target,
    "n_train": int(len(train_idx)),
    "n_test": int(len(test_idx)),
    "n_playlists_train": int(df.loc[train_idx, "playlist_id"].nunique()),
    "n_playlists_test": int(df.loc[test_idx, "playlist_id"].nunique()),
    "n_train_dates": int(df.loc[train_idx, "snapshot_date"].nunique()),
    "n_test_dates": int(df.loc[test_idx, "snapshot_date"].nunique()),
    "test_dates": [pd.Timestamp(d).date().isoformat() for d in sorted(test_dates)],
    "baseline_metrics": baseline_metrics.to_dict(orient="records"),
    "model_metrics": metrics.to_dict(orient="records"),
}
with open(metrics_path, "w") as f:
    json.dump(p, f, indent=2)

pred_path, metrics_path